# Step 3 — Value Matching (Decode Opaque Codes + Jargon)

**Databricks Genie Workshop**

In Step 2 the space couldn't answer *"revenue from PayPal"* — `payment_method` stores **opaque gateway codes** (`PM01`…`PM04`) that agent mode can't decode by profiling. Nor *"enterprise clients"* — the segment value is `Corporate`, and "enterprise" appears in no column. Nor `CA`→California (we deliberately kept no `state_name` column).

**Value matching** fixes this at the column level (`column_configs`):
- **description** — decode the codes (PM02 = PayPal, Corporate = enterprise/B2B, CA = California). *This is the lever that actually teaches meaning.*
- **synonyms** — alternate words users type.
- **entity matching** — index distinct values for fuzzy resolution.

We **round-trip**: fetch the space, add configs, patch it back. The configs live in `workshop_lib/value_matching.py`.

> **API vs UI:** `column_configs` are fully settable through the REST API — no UI step.

In [0]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.value_matching, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.value_matching, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig
from workshop_lib import genie_builder as gb
from workshop_lib import value_matching as vm
from workshop_lib import questions
# Load config saved by notebook 01 — no widgets needed.
cfg = WorkshopConfig.from_saved(spark)
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)   # created in Step 2
print("Editing Genie space:", space_id)
print(client.space_url(space_id))

## 1. Apply value matching

`vm.add_value_matching(builder, cfg)` decodes:
- **payment_method** — PM01=Credit Card, PM02=PayPal, PM03=Gift Card, PM04=Debit Card.
- **currency** — USD / CAD.
- **state_code** — CA=California, ON=Ontario, … (no state_name column exists).
- **segment** — Corporate=enterprise/B2B, SMB=small business, Consumer=individual.

Open `workshop_lib/value_matching.py` to read/edit the exact descriptions and synonyms.

In [0]:
builder = client.load_builder(space_id)     # fetch current definition
counts = vm.add_value_matching(builder, cfg) # attach value matching
print("column_configs added per table:")
for tbl, n in counts.items():
    print(f"  {tbl.split('.')[-1]}: {n}")
print(builder.summary())

## 2. Patch the space

In [0]:
client.patch_space(
    space_id,
    title=cfg.genie_space_name,
    description="Databricks Genie Workshop — with value matching.",
    warehouse_id=cfg.warehouse_id,
    serialized_space=builder.to_serialized(),
)
print("Patched.")

## 3. Verify the configs landed

In [0]:
check = client.load_builder(space_id).to_dict()
for t in check["data_sources"]["tables"]:
    if t.get("column_configs"):
        cols = [c["column_name"] for c in t["column_configs"]]
        print(t["identifier"].split(".")[-1], "->", cols)

## 4. Try it — the two question sets

In [0]:
# Questions with trap answers. Set A should work; Set B are traps
# (agent often returns the 'naive_or_trap_answer').
print(questions.render(3))
print('\n--- Trap answers from YOUR data ---')
display(questions.answer_df(spark, cfg, 3).select("set", "question", "naive_or_trap_answer", "why_agent_fails"))

## Recap & what's next

Genie now speaks the business's language for opaque codes and jargon. But it still can't compute governed **measures** consistently — what counts as "revenue" (completed only), gross **margin %**, an **on-time rate** at the right grain.

**→ Step 4** builds a **metric view** with measures, relationships, joins, and a completed-only filter.